# The reserved period: checks, then one scoring

[Open in Colab](https://colab.research.google.com/github/Gaire-commits/entropy-gate-pipeline/blob/main/notebooks/reserved_evaluation.ipynb)

Two frozen protocols share the period 2024-01-03 to 2026-10-12: `protocols/volatility_v2.yaml` and `protocols/portfolio_v2.yaml`. **It is reserved for these specifications but previously exposed through related research**; `protocols/reserved_period_disclosure.md` lists every prior use.

This notebook runs in steps. Set `STEP` below, then Runtime → Run all. Each step includes the ones before it, and finished files are kept. Results print here; nothing is downloaded. Paste the output into the chat.

| STEP | What it does | Needs | Time |
|---|---|---|---:|
| **1** | The data checks: SPY and BIL against the recorded returns, official-open coverage, whether missing opens cluster on extreme days, quoted spreads. **Scores nothing.** For the reserved period it prints coverage counts only. | Colab Secrets | about 20 minutes |
| **2** | The baseline gate on the official open, from daily bars. **No model uses PE, and nothing after 2024-01-02 is read.** | nothing more | about 10 minutes more |
| **3** | **Scores the volatility protocol on the reserved period, once,** from daily bars. The portfolio protocol was scored on 2026-11-17. It cannot be undone. | `CONFIRM` below, typed exactly | about 15 minutes more |

**Don't save this notebook to GitHub from an old copy.** The code it runs is hash-checked by `tests/test_frozen.py`, so a change to a protocol or its code shows up.

In [7]:
STEP = 0           # 0 skips the large-cap checks; 1 or 2 run them. STEP 3 has run: both protocols are scored
CONFIRM = ''       # for STEP 3 only: 'score the reserved period'
EXPLORE = ''       # after STEP 1: 'har_x', 'signals', 'pe_long', or '' for none
MIDCAP = True      # the frozen HAR-X replication on mid caps (protocols/har_x_replication_v1.yaml), about 15-20 minutes

In [8]:
import os, sys
IN_COLAB = 'google.colab' in sys.modules
REPO = '/content/entropy-gate-pipeline' if IN_COLAB else os.environ.get('PORTFOLIO_REPO', os.getcwd())
if IN_COLAB:
    if os.path.isdir(f'{REPO}/.git'):
        !cd {REPO} && git pull -q --ff-only
    else:
        !git clone -q https://github.com/Gaire-commits/entropy-gate-pipeline.git {REPO}
    %cd {REPO}
    !git log --oneline -1
    from google.colab import userdata
    try:
        with open('.env', 'w') as f:
            f.write(f"APCA_API_KEY_ID={userdata.get('APCA_API_KEY_ID')}\nAPCA_API_SECRET_KEY={userdata.get('APCA_API_SECRET_KEY')}\n")
        print('keys written to a local .env (never printed, never committed)')
    except Exception as exc:
        print('No Alpaca keys in Colab Secrets:', type(exc).__name__, '- add them from the key icon, then run this cell again.')
else:
    os.chdir(REPO)
for m in [m for m in sys.modules if m == 'src' or m.startswith('src.')]:
    del sys.modules[m]
sys.path[:0] = [REPO, f'{REPO}/scripts']

from pathlib import Path
REF = os.environ.get('RESERVED_REF', '/content/reserved/reference')
WORK = os.environ.get('RESERVED_WORK', '/content/reserved')
DRIVE = '/content/drive/MyDrive/entropy-gate-pipeline-data/data'
Path(REF).mkdir(parents=True, exist_ok=True)

/content/entropy-gate-pipeline
17adb5d (HEAD -> main, origin/main, origin/HEAD) Freeze the HAR-X model and its mid-cap replication before any mid-cap data is loaded
keys written to a local .env (never printed, never committed)


## Step 1: the data checks

In [9]:
if STEP >= 1:
    !python scripts/fetch_reference.py --universe universe/sp500.csv --start 2016-01-04 --end 2026-10-13 --out {REF} --parts daily,corporate_actions,meta,auctions 2>&1 | tail -4
    !python scripts/official_prices.py --reference {REF}
    !python scripts/protocol_checks.py --reference {REF}
else:
    print('STEP is 0 - the large-cap checks were skipped')

STEP is 0 - the large-cap checks were skipped


## Step 2: the baseline gate on the official open

In [10]:
if STEP >= 2:
    # daily bars only (protocols/amendment_4.md): no Drive, no 5-minute cache
    !python scripts/vol_forecast.py --stage baseline --reference {REF} --official {REF}/official_prices.parquet 2>&1 | grep -v Warning
else:
    print('STEP is', STEP, '- skipped')

STEP is 0 - skipped


## Step 3: locked

This cell scores the reserved period, once. It does nothing unless `STEP = 3` and `CONFIRM` is typed exactly, and it is not to run until Kshitiz decides. Record the outcome in `RESULTS.md` whatever it is.

In [11]:
if STEP >= 3:
    assert CONFIRM == 'score the reserved period', "set CONFIRM exactly to run this step"
    # The portfolio protocol was scored on 2026-11-17 and is not run again. Volatility on daily bars only (protocols/amendment_4.md).
    print('=' * 30, 'VOLATILITY')
    !python scripts/vol_forecast.py --stage reserved --score-reserved --reference {REF} --official {REF}/official_prices.parquet 2>&1 | grep -v Warning
else:
    print('STEP is', STEP, '- the reserved period was not scored')

STEP is 0 - the reserved period was not scored


## Exploration (not registered tests)

Both protocols are scored, so the data from 2016 to 2026 is open for exploring. Anything found is a lead: it is written as a rule and tested on trading days after 2026-11-17. Every test run is counted. Daily bars only, no Drive.

- `'signals'`: ten simple daily features against the next day's return, the next day's open-to-close return, the next two days' return, and next-day variance beyond what volatility history predicts. About 40 tests, with a false-discovery correction.
- `'pe_long'`: PE once the long-run volatility level is in the model (run 2026-11-17; recorded in RESULTS.md).
- `'har_x'`: HAR plus the long-run level, then adding the signal search's features in blocks (the range against its usual level, recent returns, the largest return, volume shock), each measured out of sample, 2017 to 2026.

In [12]:
if EXPLORE == 'har_x':
    !python scripts/explore_har_x.py --reference {REF} --official {REF}/official_prices.parquet 2>&1 | grep -v Warning
elif EXPLORE == 'signals':
    !python scripts/explore_signals.py --reference {REF} --official {REF}/official_prices.parquet 2>&1 | grep -v Warning
elif EXPLORE in ('pe_long', True):
    !python scripts/explore_pe_long.py --reference {REF} --official {REF}/official_prices.parquet 2>&1 | grep -v Warning
else:
    print('EXPLORE is off')

EXPLORE is off


## Replication: the volatility model on mid caps (frozen 2026-11-18)

The model found on large caps (HAR plus the long-run level, recent returns and volume) is frozen in `protocols/har_x_replication_v1.yaml` and run once on the S&P MidCap 400, whose daily volatility has not been used for anything. It downloads the mid caps' daily bars and auction prints, takes the official open, and judges the model by the protocol's rule. Daily bars only, no Drive. Paste the output into the chat.

In [13]:
if MIDCAP:
    MID = '/content/midcap/reference'
    !python scripts/fetch_reference.py --universe universe/sp400.csv --start 2016-01-04 --end 2026-10-13 --out {MID} --parts daily,corporate_actions,meta,auctions 2>&1 | tail -4
    !python scripts/official_prices.py --reference {MID} 2>&1 | grep -A14 "coverage by year"
    !python scripts/har_x_replication.py --reference {MID} --official {MID}/official_prices.parquet 2>&1 | grep -v Warning
else:
    print('MIDCAP is off')

wrote /content/midcap/reference/daily/daily_sip_split_nomap.parquet: 984,150 bars, 400 of 400 symbols
wrote /content/midcap/reference/corporate_actions.parquet: 10,822 corporate actions (33 older symbols added)
wrote /content/midcap/reference/auctions.parquet: 4,995,943 auction prints, 400 of 400 symbols
done in 525s, 1244 requests; manifest at /content/midcap/reference/manifest.json
coverage by year (shares of vendor bars; counts only, so every year is shown):
        bars  with_listing  with_official_open  with_official_close
date                                                               
2016   82643        0.9993              0.9959               0.9977
2017   85933        0.9999              0.9968               0.9983
2018   87908        1.0000              0.9968               0.9988
2019   90026        0.9996              0.9980               0.9964
2020   92438        0.9990              0.9984               0.9979
2021   95434        1.0000              0.9997            